In [16]:
import pickle
import numpy as np
import pandas as pd
import os

In [25]:
class LegacyUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        renames = {
            "pandas.indexes.base": "pandas.core.indexes.base",
            "pandas.indexes.numeric": "pandas.core.indexes.numeric",
            "pandas.indexes.range": "pandas.core.indexes.range",
            "pandas.indexes.multi": "pandas.core.indexes.multi",
            "pandas.indexes.category": "pandas.core.indexes.category",
            "pandas.indexes.datetimes": "pandas.core.indexes.datetimes",
            "pandas.indexes.period": "pandas.core.indexes.period",
            "pandas.indexes.timedeltas": "pandas.core.indexes.timedeltas",
        }
        module = renames.get(module, module)
        return super().find_class(module, name)

with open("data/raw/LSWMD.pkl", "rb") as f:
    df = LegacyUnpickler(f, encoding="latin1").load()

/var/folders/42/dqcm_7xd4pz6qhnbk1lrgcyr0000gn/T/ipykernel_41570/1173136058.py:17: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  df = LegacyUnpickler(f, encoding="latin1").load()


In [26]:
df['failureType'] = df['failureType'].apply(
    lambda x: x[0][0] if isinstance(x, np.ndarray) and x.size > 0 else np.nan
)
df['failureType'] = df['failureType'].astype(str)
print(df['failureType'].value_counts(dropna=False))

failureType
NaN          638507
none         147431
Edge-Ring      9680
Edge-Loc       5189
Center         4294
Loc            3593
Scratch        1193
Random          866
Donut           555
Near-full       149
Name: count, dtype: int64


In [27]:
df = df[df['failureType'].notna()]
df = df[df['failureType'] != 'none']
print(df.shape)
print(df['failureType'].value_counts())

(25519, 6)
failureType
Edge-Ring    9680
Edge-Loc     5189
Center       4294
Loc          3593
Scratch      1193
Random        866
Donut         555
Near-full     149
Name: count, dtype: int64


In [29]:
df['waferMap'] = df['waferMap'].apply(lambda x: x / 2.0)
print(np.unique(df['waferMap'].iloc[0]))

[0.   0.25 0.5 ]


In [31]:
df['label'] = pd.Categorical(df['failureType']).codes
categories = pd.Categorical(df['failureType']).categories
for i, cat in enumerate(categories):
    print(f"{i}: {cat}")

0: Center
1: Donut
2: Edge-Loc
3: Edge-Ring
4: Loc
5: Near-full
6: Random
7: Scratch


In [32]:
os.makedirs("data/processed", exist_ok=True)
df.to_pickle("data/processed/clean.pkl")
print(f"Saved {df.shape[0]} wafers to data/processed/clean.pkl")

Saved 25519 wafers to data/processed/clean.pkl
